# Лабораторная работа 5: Проектирование систем потоковой обработки и данных

## Введение

- **Вариант:** последняя цифра зачётки — **8**
- **Датасет:** MovieLens ml-32m (из ЛР-1; рабочий срез `lab2/work/ratings_100mb.parquet`, 4 млн оценок)
- **Бизнес-сценарий:** рекомендательная система онлайн-кинотеатра (сквозная линия ЛР-1…ЛР-4)
- **Уникальное требование (цифра 8):** задержка **p99 < 45 мс** — обязательно к выполнению
- **Технология потоковой обработки:** Flink 1.20 (PyFlink, true streaming)

Стенд целиком поднимается одной командой:

```bash
cd lab5 && docker compose up -d --build
docker compose logs -f monitor     # блок ## METRICS ##
```

Задание состоит из 8 частей — ниже они разобраны по порядку.


## Часть 1. Проектирование потоковой системы

### 1.1. Micro-batch или true streaming

| Критерий | Micro-batch (Spark Structured Streaming) | True streaming (Flink) |
|---|---|---|
| Минимальная задержка | интервал триггера, десятки–сотни мс | единицы мс на событие |
| Стоимость | дешевле по CPU (батчи эффективнее) | дороже: обработка поштучно |
| Late-arriving | watermark + `dropDuplicates` | watermark + `allowedLateness` + side output |

**Выбор: true streaming на Flink.** Требование p99 < 45 мс отсекает micro-batch: даже при
`Trigger.ProcessingTime(0)` к задержке добавляется планирование задач и накопление батча.

### 1.2. Архитектура системы

```mermaid
graph LR
    A[Источники данных<br/>MovieLens-оценки] -->|produce| B[Kafka<br/>rating_events]
    B -->|KafkaSource| C{Flink<br/>true streaming}
    C -->|keyBy userId| D[Stateful инференс<br/>ONNX Runtime]
    E[(Feature Store<br/>Redis)] -.->|фичи| D
    F[MLflow<br/>model registry] -.->|recommender.onnx:v1| D
    D -->|predictions| G[Kafka predictions]
    C -->|keyBy movieId, окно 1 мин| H[Агрегация]
    H -->|movie_stats| G2[Kafka movie_stats]
    C -.->|late-arriving| I[(dlq_late)]
    C -.->|ошибки| J[(dlq_errors)]
    G --> K[monitor: перцентили, PSI]
    K --> L[Prometheus]
    L --> M[Grafana + алерты]
```

### 1.3. Матрица требований и решений

| Требование | Решение | Обоснование |
|---|---|---|
| Задержка p99 < 45 мс | Flink true streaming, ONNX в процессе, тюнинг Kafka/PyFlink | см. Часть 7 |
| Обработка всех событий | checkpointing + идемпотентный продюсер + дедуп по `event_id` | Часть 5 |
| Late-arriving / out-of-order | watermark 5 с + `allowedLateness` 5 мин + side output | Часть 2 |
| Интеграция с ML | Redis Feature Store + ONNX Runtime, версия в MLflow | Часть 3 |
| Мониторинг и алертинг | Flink Prometheus reporter + Grafana + PSI | Часть 6 |
| Отказоустойчивость | restart-strategy, checkpointing, DLQ | Часть 5 |

### 1.4. Допустимая задержка и стратегия late-arriving

- допускаем опоздание **5 минут** (`allowedLateness`), watermark — **5 секунд**;
- стратегия: `watermark + allowedLateness + side outputs` — событие, опоздавшее больше
  чем на 5 минут, не теряется, а уходит в `dlq_late` для пакетной дообработки.


## Часть 2. Реализация потоковой обработки

### 2.1. Инфраструктура

`docker-compose.yml` поднимает: Kafka в режиме **KRaft** (без ZooKeeper), Redis,
Flink (jobmanager + taskmanager + отправка задания), model server, генератор нагрузки,
Prometheus и Grafana. Kafka слушает на двух интерфейсах: `kafka:29092` для контейнеров
и `localhost:9092` для хоста.

### 2.2. Потоковое задание

Ядро — `flink/jobs/recommender_stream.py`. Ключевые места:


In [ ]:
# Фрагмент flink/jobs/recommender_stream.py — окна, watermark, allowedLateness

from pyflink.common.time import Time
from pyflink.datastream.window import TumblingEventTimeWindows

watermarked = parsed.assign_timestamps_and_watermarks(
    WatermarkStrategy.for_bounded_out_of_orderness(Duration.of_seconds(WATERMARK_SEC))
    .with_timestamp_assigner(EventTimestampAssigner())      # event_time -> внутренний ts
)

# ветка 1: stateful инференс (keyBy user_id + ValueState с TTL)
scored = watermarked.key_by(lambda e: e[1], key_type=Types.LONG()) \
                     .process(UserStateInferenceFunction(), output_type=Types.STRING())

# ветка 2: оконная агрегация + обработка late-arriving
windowed = (
    watermarked.key_by(lambda e: e[2], key_type=Types.LONG())
    .window(TumblingEventTimeWindows.of(Time.seconds(WINDOW_SEC)))
    .allowed_lateness(ALLOWED_LATENESS_SEC * 1000)          # ВАЖНО: миллисекунды
    .side_output_late_data(LATE_TAG_WINDOW)
    .aggregate(RatingAggregate(), MovieStatsWindow(),
               accumulator_type=ACC_TYPE, output_type=Types.STRING())
)

### 2.3. State management

`UserStateInferenceFunction` — `KeyedProcessFunction` с `ValueState`
`[rating_count, rating_sum, last_event_ms, last_rating]` и TTL 24 часа.
TTL обязателен: без него state растёт неограниченно, checkpoint раздувается и
пропускная способность падает.


In [ ]:
# Фрагмент: keyed ValueState с TTL
ttl = (StateTtlConfig.new_builder(Time.hours(24))
       .set_update_type(StateTtlConfig.UpdateType.OnCreateAndWrite)
       .set_state_visibility(StateTtlConfig.StateVisibility.NeverReturnExpired)
       .build())

desc = ValueStateDescriptor("user_state", Types.LIST(Types.DOUBLE()))
desc.enable_time_to_live(ttl)
self.state = runtime_context.get_state(desc)

# в process_element:
st = self.state.value() or [0.0, 0.0, 0.0, 0.0]
st[0] += 1.0                 # число оценок пользователя
st[1] += float(rating)       # сумма оценок
st[3] = float(rating)        # последняя оценка
self.state.update(st)

### 2.4. Checkpointing (exactly-once)

```python
env.enable_checkpointing(CHECKPOINT_MS, CheckpointingMode.EXACTLY_ONCE)
cc = env.get_checkpoint_config()
cc.set_min_pause_between_checkpoints(10000)
cc.set_checkpoint_timeout(120000)
cc.set_max_concurrent_checkpoints(1)
cc.set_tolerable_checkpoint_failure_number(3)
cc.set_checkpoint_storage_dir("file:///opt/flink/checkpoints")
env.set_restart_strategy(RestartStrategies.fixed_delay_restart(3, 1000))
```

Интервал 30 с (а не 1 с) выбран осознанно: частые checkpoint'ы блокируют источник
барьерами и добавляют задержку. Гарантия при этом не теряется — см. Часть 5.

### Вывод по Части 2

Реализованы окна на event-time, watermark, `allowedLateness`, side outputs,
keyed state с TTL, checkpointing и DLQ. Отдельная тонкость PyFlink: `allowed_lateness`
принимает **миллисекунды**, а `OutputTag` нельзя создавать с уже «прокинутым» в JVM
`TypeInformation` — иначе оператор не сериализуется.


## Часть 3. Интеграция с ML-моделью

### 3.1. Обучение и экспорт в ONNX

Задача — бинарная классификация: поставит ли пользователь высокую оценку (≥ 4.0).
Сплит временной (80/20): обучаемся на прошлом, проверяем на будущем. Статистики
пользователей и фильмов считаются **только на train** — иначе утечка.


In [ ]:
# model/train_export.py (сокращённо): обучение + экспорт ONNX + версия в MLflow
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from skl2onnx import to_onnx
from skl2onnx.common.data_types import FloatTensorType

FEATURE_NAMES = ["user_avg_rating", "user_rating_count", "user_std_rating",
                 "movie_avg_rating", "movie_rating_count", "movie_year",
                 "hour", "day_of_week"]

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
model.fit(X_train, y_train)

# target_opset фиксируем: onnxruntime 1.20 (в образе Flink) не умеет opset 22
onnx_model = to_onnx(model,
                     initial_types=[("input", FloatTensorType([None, len(FEATURE_NAMES)]))],
                     options={"zipmap": False},
                     target_opset=17)
open("models/recommender.onnx", "wb").write(onnx_model.SerializeToString())

LogisticRegression: AUC=0.6591 F1=0.4870 (порог=0.373)
XGBoost:            AUC=0.6609 F1=0.5664
Для потока выбрана: LogisticRegression
ONNX == sklearn: max|diff|=8.94e-08; инференс p50=0.03 мс p99=0.18 мс
Фиче-стор выгружен: work/feature_store.json (5.4 МБ)
MLflow run: f91389dfa49e4c1792bd3c736ab5e38b


### 3.2. Feature Store на Redis

```
feat:user:<userId>   → user_avg_rating, user_rating_count, user_std_rating
feat:movie:<movieId> → movie_avg_rating, movie_rating_count, movie_year
feat:global          → средние для холодного стартера
```

Загружено **21 434** пользователя и **29 194** фильма (50 629 ключей).
Режимы: `warm` (выгрузка в память воркера, 0.02 мс на событие) и `online`
(`HGETALL` на каждое событие, +0.2–0.5 мс). Если Redis недоступен — читается снапшот JSON.

### 3.3. Online-инференс в потоке

```python
vec = np.array([[u_avg, u_cnt, u_std, m_avg, m_cnt, m_year, hour, dow]], dtype=np.float32)
score = float(np.asarray(self.model.run(None, {"input": vec})[self.prob_idx])[:, 1][0])
```

`prob_idx` ищется по имени выхода `probabilities`: у sklearn-модели выходов два
(`label` и `probabilities`), и обращение по индексу 0 даёт метку, а не вероятность.

### 3.4. Оптимизация под низкую задержку

| Приём | Эффект |
|---|---|
| ONNX вместо pickle-sklearn | инференс p99 = 4.09 мс в потоке |
| Логистическая регрессия вместо XGBoost | AUC −0.002, latency в разы ниже |
| `intra_op_num_threads=1` | нет переключений контекста на маленьком батче |
| Кэш фич в памяти воркера | 0.02 мс вместо 0.2–0.5 мс |
| Отдельный model server (FastAPI) | точка версионирования и HTTP-инференс для сверки |

### Вывод по Части 3

Модель обучается на реальном MovieLens, экспортируется в ONNX (проверена
эквивалентность sklearn ↔ ONNX), фичи берутся из Redis, версия зафиксирована в MLflow.
Инференс укладывается в бюджет с запасом в сотни раз.


## Часть 4. Предикты на лету из Kafka

Поток: `rating_events` → обогащение фичами → ONNX-инференс → `predictions`.

- генератор (`load_generator/generator.py`) пишет реальные оценки MovieLens и
  штампует `sent_ts` для замера end-to-end задержки;
- результат содержит `score`, `will_like`, `model_version`, `infer_ts`, `feat_ms`, `infer_ms`;
- идемпотентность обеспечивается ключом `event_id` (дедупликация на потребителе).

Пример реального предикта:

```json
{"event_id": "b32dc174b42a40b79d68a4d778a873b6", "user_id": 1411, "movie_id": 1837,
 "rating": 3.5, "score": 0.12469, "will_like": false, "model_version": "recommender.onnx:v1",
 "sent_ts": 1795247669798.7, "infer_ts": 1795247677831.6, "is_late": true,
 "u_avg": 3.4967, "m_avg": 2.7321, "feat_ms": 0.0185, "infer_ms": 0.5338}
```

### Замер end-to-end задержки

`monitoring/monitor.py` для каждого `event_id` считает два числа:

- **end-to-end** — от отправки в топик до появления предикта в `predictions`;
- **inference** — от отправки до штампа `infer_ts` внутри Flink.


## Часть 5. Обработка ошибок и fault tolerance

### 5.1. Exactly-once на трёх уровнях

1. **Checkpointing** `EXACTLY_ONCE` — состояние и offset'ы Kafka восстанавливаются вместе.
2. **Идемпотентный продюсер** (`enable.idempotence=true`) — дубликатов при ретраях нет.
3. **Дедупликация по `event_id`** на потребителе — приём, прямо разрешённый заданием
   («запись с дедупликацией по `tx_id` … результат проверки идемпотентности тот же»).

Транзакционный sink (`DeliveryGuarantee.EXACTLY_ONCE`) сохранён и включается переменной
`SINK_GUARANTEE=exactly-once`. По умолчанию он выключен осознанно: транзакция делает
запись видимой только в момент коммита, то есть на checkpoint'е, что добавляет к
end-to-end задержке ~интервал checkpoint'а.

### 5.2. DLQ

- `dlq_errors` — битые записи и исключения обработки (side output `ERROR_TAG`);
- `dlq_late` — события, опоздавшие больше чем на `allowedLateness` (side output окна).

Тонкость PyFlink: side output по умолчанию сериализуется как `PICKLED_BYTE_ARRAY`
(`byte[]` в JVM), а `KafkaSink` с `SimpleStringSchema` ждёт `String` — нужен
промежуточный `.map(side_output_to_str, output_type=Types.STRING())`.

### 5.3. Проверка

На прогонах 800 и 1200 событий: `DUPLICATES=0`, `ERRORS=0`, checkpoint 0.04–0.16 с.

### Вывод по Части 5

Гарантия «эффективно однократной» доставки обеспечена; сбои обрабатываются
restart-strategy + восстановлением из checkpoint, ошибочные данные не роняют поток.


## Часть 6. Мониторинг и алертинг

### 6.1. Метрики

Flink отдаёт свои метрики в Prometheus (`metrics.reporter.prom`, порты 9250/9251).
Бизнес-метрики публикует `monitoring/monitor.py` на `:8001/metrics`:
p50/p95/p99 задержки, пропускная способность, late-события, ошибки, длительность
checkpoint, PSI-дрейф, счётчик дубликатов.

### 6.2. Алерты (`monitoring/prometheus/alerts.yml`)

| Алерт | Условие | Severity |
|---|---|---|
| `HighStreamingLatencyP99` | `lab5_latency_p99_ms > 45` | critical |
| `HighStreamingLatencyP95` | `lab5_latency_p95_ms > 30` | warning |
| `HighKafkaLag` | `kafka_consumergroup_lag > 1000` | critical |
| `HighErrorRate` | `rate(lab5_errors_total[1m]) > 5` | warning |
| `DataDriftDetected` | `lab5_data_drift_psi > 0.2` | warning |
| `CheckpointFailing` | рост `numberOfFailedCheckpoints` | critical |

### 6.3. Дрейф данных (PSI)

PSI считается по фиче `user_avg_rating`: онлайн-распределение против offline-снапшота
фиче-стора (21 434 пользователя). На прогоне **PSI = 0.2455** > 0.2 → дрейф зафиксирован
(для синтетического потока из среза датасета это ожидаемо).

### 6.4. Дашборд Grafana

`monitoring/grafana/dashboards/streaming.json`, три панели:

1. **Состояние потока** — p50/p95/p99, SLA, пропускная способность, дубликаты;
2. **Качество данных** — PSI, late-arriving события, ошибки в DLQ;
3. **Состояние state** — длительность и размер checkpoint, задержка инференса.

### Вывод по Части 6

Мониторинг покрывает все три требуемые группы (производительность, качество данных,
состояние state), алерты настроены, дрейф детектируется.


## Часть 7. Оптимизация под задержку

### 7.1. Разбор по этапам

Замеры с разделением этапов (серия из 40 событий, `infer_ts - sent_ts` и `recv - infer_ts`):

| Конфигурация | source + обработка | sink + потребитель |
|---|---|---|
| по умолчанию | ~4000 мс | ~50 мс |
| `bundle.size=50, bundle.time=10 мс` | **~370 мс** | ~50 мс |

### 7.2. Найденные узкие места

1. **«Бандлы» Python-воркера PyFlink.** По умолчанию воркер копит 1000 элементов
   **или 1000 мс** и только потом обрабатывает. В цепочке четыре Python-оператора
   (`parse` → timestamp assigner → key selector → `KeyedProcessFunction`), то есть до
   4 секунд постоянной добавки. Настройки читаются **на клиенте** при построении графа
   задания: их нужно задавать в контейнере `flink-submit` и передавать через
   `env.configure(Configuration())` — в `FLINK_PROPERTIES` jobmanager'а их недостаточно.
2. **`fetch.max.wait.ms` Kafka-консьюмера** — по умолчанию до 500 мс ожидания батча.
   Установлено `fetch.max.wait.ms=5`, `fetch.min.bytes=1`.
3. **Видимость транзакционного sink'а** — запись появляется только на коммите
   транзакции (см. Часть 5).
4. **Пропускная способность Python DataStream API.** Уменьшение бандла снижает
   задержку, но повышает число gRPC-вызовов на событие: устойчивая пропускная
   способность ~30 событий/с.

### 7.3. Итоговый замер (800 событий, 40 соб/с, 5 % late)

```
## METRICS ##
REQUIRED_LATENCY=45
MEASURED_P50_LATENCY=234.42
MEASURED_P95_LATENCY=1284.83
MEASURED_P99_LATENCY=2721.67
PIPELINE_P99_LATENCY=2705.90
SLA_OK=False
LATE_DATA_HANDLED=True
LATE_EVENTS=60
PREDICTIONS=799
DUPLICATES=0
ERRORS=0
THROUGHPUT_EPS=28.8
CHECKPOINT_DURATION_SEC=0.03
DATA_DRIFT_SCORE=0.2437
```

### 7.4. Вывод: требование p99 < 45 мс не выполнено

Честный результат:

- **инференс** укладывается в бюджет с запасом: p99 = **0.18 мс**;
- **end-to-end через PyFlink Python DataStream API** — нет: после тюнинга остаётся
  ~370 мс на событие, под нагрузкой добавляется очередь.

Причина не в архитектуре (Flink + event-time + ONNX выбраны верно), а в
**Python DataStream API**: каждый Python-оператор — это gRPC-хоп в отдельный
Python-воркер (Beam FnApi) со своим буфером.

**Что нужно для 45 мс в production:**

1. Перенести разбор, watermark и оконную агрегацию в **Flink SQL / Java-операторы** —
   Python останется только на инференс (один хоп вместо четырёх).
2. Либо вынести инференс в **model server** и вызывать асинхронно (`AsyncDataStream`).
3. Поднять параллелизм источника (партиции топика ≥ параллелизма задачи).
4. Если micro-batch допустим — Spark Structured Streaming + ONNX в `foreachBatch`
   (как в Части 4 задания), с той же оговоркой по бюджету.


## Часть 8. Тестирование и валидация

### 8.1. Генератор нагрузки

`load_generator/generator.py`: режимы `normal`, `peak` (×2), `late`, `burst`;
пишет реальные оценки MovieLens, штампует `sent_ts`, считает свою пропускную способность.

### 8.2. Автоматические проверки

```bash
docker compose run --rm tests          # pytest: задержка + late-arriving
docker compose run --rm tests python check_requirements.py
```


In [ ]:
# Сводная проверка требований (tests/check_requirements.py)
import json

m = json.load(open("work/metrics.json", encoding="utf-8"))
req = m["required_latency_ms"]

checks = [
    ("1. Задержка p99 < 45 мс",              m["measured_p99_latency_ms"] < req),
    ("2a. Поток обрабатывается",              m["predictions"] > 0),
    ("2b. Late-arriving обработаны",          m["late_data_handled"]),
    ("2c. Пропускная способность > 5 соб/с",  m["throughput_eps"] > 5),
    ("3a. Онлайн-инференс измерен",           m["inference_p99_ms"] > 0),
    ("4a. Дубликатов нет",                    m["duplicates"] == 0),
    ("4b. Checkpoint'ы выполняются",          m["checkpoint_duration_sec"] > 0),
    ("4c. Доля ошибок < 1%",                  m["error_rate"] < 0.01),
    ("5a. PSI посчитан",                      m["data_drift_psi"] >= 0.0),
]
for name, ok in checks:
    print(f"[{'PASS' if ok else 'FAIL'}] {name}")

print(f"\nИТОГО: {sum(ok for _, ok in checks)}/{len(checks)} проверок пройдено")

[FAIL] 1. Задержка p99 < 45 мс
[PASS] 2a. Поток обрабатывается
[PASS] 2b. Late-arriving обработаны
[PASS] 2c. Пропускная способность > 5 соб/с
[PASS] 3a. Онлайн-инференс измерен
[PASS] 4a. Дубликатов нет
[PASS] 4b. Checkpoint'ы выполняются
[PASS] 4c. Доля ошибок < 1%
[PASS] 5a. PSI посчитан

ИТОГО: 8/9 проверок пройдено


### 8.3. Нагрузочные режимы

| Режим | Подача | Результат |
|---|---|---|
| `normal` | 40–60 соб/с | все события обработаны, 0 дубликатов, 0 ошибок |
| `peak` | ×2 от нормальной | очередь растёт, p99 деградирует — узкое место в Части 7 |
| `late` | 5 % событий на 6 мин в прошлом | все late-события учтены, `LATE_DATA_HANDLED=True` |

### Вывод по Части 8

Все функциональные требования подтверждены на живом стенде. Не выполнено только
уникальное требование по задержке — разбор и план исправления в Части 7.


# Заключение

В ходе лабораторной работы построена система потоковой обработки для рекомендательной
системы онлайн-кинотеатра на датасете MovieLens (ЛР-1).

**Реализовано и проверено на живом стенде:**

1. **Потоковая обработка** — окна 1 минута на event-time, watermark 5 с,
   `allowedLateness` 5 мин, side outputs (2.5 балла).
2. **Интеграция с ML** — Feature Store на Redis, online-инференс ONNX в потоке,
   версия модели в MLflow, задержка инференса p99 = 4.09 мс в потоке (2.0 балла).
3. **Exactly-once** — checkpointing `EXACTLY_ONCE`, идемпотентный продюсер,
   дедупликация по `event_id`; 0 дубликатов на 2000 событий (1.5 балла).
4. **Мониторинг и алертинг** — метрики в Prometheus, 6 алертов, дашборд Grafana
   из трёх панелей, детекция дрейфа по PSI (2.0 балла).
5. **Оптимизация под задержку** — найдены и устранены три узких места,
   задержка обработки снижена с ~4000 до ~370 мс (1.5 балла).
6. **Отчётность** — README, ноутбук, автоматические проверки (0.5 балла).

**Не достигнуто:** end-to-end p99 < 45 мс. Причина — накладные расходы Python
DataStream API PyFlink (gRPC-хоп и буфер на каждый Python-оператор). План исправления:
перенести разбор/watermark/агрегацию в Flink SQL или Java, оставив Python только
на инференс.
